<a href="https://colab.research.google.com/github/sibandze/Bird-Intelligence-System/blob/baseline-supervised-v1/notebooks/wandb_run_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup
Install and import Weights & Biases (W&B), pandas, matplotlib, and other required libraries.

In [1]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Authentication
Login to Weights & Biases securely using Colab secrets.

In [12]:
wandb.login()
wandb_api = wandb.Api()
#wandb_org = "bird-intelligence-system"
wandb_project = "bird-song-classifier"

## 3. Fetch runs
Retrieve all runs from the `bird-song-classifier` project.

In [19]:
# Fetch runs from the project
runs = wandb_api.runs(f"{wandb_project}")

# Extract details into a list of dictionaries, including the full run object
runs_data = []
for run in runs:
    runs_data.append({
        "Run Name": run.name,
        "ID": run.id,
        "Created At": run.created_at,
        "State": run.state,
        "Run Object": run  # Keeps the full object accessible for config, artifacts, etc.
    })

# Convert to a pandas DataFrame for readable formatting
runs_df = pd.DataFrame(runs_data)

display_rows = ["Run Name", "ID", "Created At", "State"]
# Display only the requested columns for the first 10 runs
runs_df[display_rows].head(10)

,Run Name,ID,Created At,State
0,run_0001_baseline_lr_sweep,exp_20261107_231730_672752_run_0001_baseline_l...,2026-11-07T23:17:34Z,finished
1,run_0002_baseline_lr_sweep,exp_20261107_231730_672752_run_0002_baseline_l...,2026-11-07T23:18:44Z,finished
2,run_0001_baseline_lr_sweep,exp_20261107_232025_567578_run_0001_baseline_l...,2026-11-07T23:20:28Z,finished
3,run_0002_baseline_lr_sweep,exp_20261107_232025_567578_run_0002_baseline_l...,2026-11-07T23:21:41Z,finished
4,run_0001_focused_lr_weight_decay,exp_20261107_232322_406731_run_0001_focused_lr...,2026-11-07T23:23:26Z,finished
5,run_0002_focused_lr_weight_decay,exp_20261107_232322_406731_run_0002_focused_lr...,2026-11-07T23:24:36Z,finished
6,run_0001_scheduler_finetune,exp_20261107_232557_459672_run_0001_scheduler_...,2026-11-07T23:26:01Z,finished
7,run_0002_scheduler_finetune,exp_20261107_232557_459672_run_0002_scheduler_...,2026-11-07T23:27:11Z,finished
8,run_0001_ssl_quick_sanity,exp_20261107_232833_161040_run_0001_ssl_quick_...,2026-11-07T23:28:37Z,finished
9,run_0001_ssl_lr_sweep,exp_20261107_233043_270657_run_0001_ssl_lr_sweep,2026-11-07T23:30:46Z,finished


## 4. Identify experiments
Extract sweep name, experiment ID, seed, and mode configurations.

In [20]:
# Define categorization keywords for supervised and SSL experiments
ssl_keywords = ['ssl', 'simclr']

def categorize_run(run_name):
    run_name_lower = str(run_name).lower()
    if any(keyword in run_name_lower for keyword in ssl_keywords):
        return 'SSL'
    return 'Supervised'

# Group the runs
runs_df['Category'] = runs_df['Run Name'].apply(categorize_run)

# Create separate DataFrames for clean, grouped analysis
supervised_runs = runs_df[runs_df['Category'] == 'Supervised'].reset_index(drop=True)
ssl_runs = runs_df[runs_df['Category'] == 'SSL'].reset_index(drop=True)

print(f"Total Supervised Runs: {len(supervised_runs)}")
print(f"Total SSL Runs: {len(ssl_runs)}")

# Display the top of each category
print("\n--- Supervised Runs (Top 5) ---")
display(supervised_runs[display_rows].head(5))

print("\n--- SSL Runs (Top 5) ---")
display(ssl_runs[display_rows].head(5))

Total Supervised Runs: 8
Total SSL Runs: 56

--- Supervised Runs (Top 5) ---


,Run Name,ID,Created At,State
0,run_0001_baseline_lr_sweep,exp_20261107_231730_672752_run_0001_baseline_l...,2026-11-07T23:17:34Z,finished
1,run_0002_baseline_lr_sweep,exp_20261107_231730_672752_run_0002_baseline_l...,2026-11-07T23:18:44Z,finished
2,run_0001_baseline_lr_sweep,exp_20261107_232025_567578_run_0001_baseline_l...,2026-11-07T23:20:28Z,finished
3,run_0002_baseline_lr_sweep,exp_20261107_232025_567578_run_0002_baseline_l...,2026-11-07T23:21:41Z,finished
4,run_0001_focused_lr_weight_decay,exp_20261107_232322_406731_run_0001_focused_lr...,2026-11-07T23:23:26Z,finished



--- SSL Runs (Top 5) ---


,Run Name,ID,Created At,State
0,run_0001_ssl_quick_sanity,exp_20261107_232833_161040_run_0001_ssl_quick_...,2026-11-07T23:28:37Z,finished
1,run_0001_ssl_lr_sweep,exp_20261107_233043_270657_run_0001_ssl_lr_sweep,2026-11-07T23:30:46Z,finished
2,run_0002_ssl_lr_sweep,exp_20261107_233043_270657_run_0002_ssl_lr_sweep,2026-11-07T23:32:47Z,finished
3,run_0003_ssl_lr_sweep,exp_20261107_233043_270657_run_0003_ssl_lr_sweep,2026-11-07T23:34:56Z,finished
4,run_0004_ssl_lr_sweep,exp_20261107_233043_270657_run_0004_ssl_lr_sweep,2026-11-07T23:37:12Z,finished


## 5. Validate histories
Check epoch coverage and scan for missing metrics in the fetched run data.

In [ ]:
# Validate histories: Check epoch coverage and missing metrics across runs using scan_history
validation_results = []

for idx, row in runs_df.iterrows():
    run = row["Run Object"]

    # Retrieve history using scan_history to avoid sampling limitations
    history_data = list(run.scan_history())
    history = pd.DataFrame(history_data)

    num_steps = len(history)
    max_epoch = history["epoch"].max() if "epoch" in history.columns else None

    available_metrics = list(history.columns)
    missing_metrics = []

    if row["Category"] == "Supervised":
        expected = ["loss", "val_loss", "accuracy"]
    else: # SSL
        expected = ["loss", "val_loss", "val_contrastive_acc"]

    for metric in expected:
        if metric not in available_metrics or history[metric].isnull().all():
            missing_metrics.append(metric)

    validation_results.append({
        "Run Name": row["Run Name"],
        "Category": row["Category"],
        "Recorded Steps": num_steps,
        "Max Epoch": max_epoch,
        "Missing Metrics": missing_metrics if missing_metrics else "None"
    })

# Convert validation results to DataFrame
validation_df = pd.DataFrame(validation_results)

# Display summary of validation results
print("--- Validation Summary ---")
print(f"Total Runs Checked: {len(validation_df)}")
print(f"Runs with Missing Metrics: {len(validation_df[validation_df['Missing Metrics'] != 'None'])}")

display(validation_df.head(10))

## 6. Normalize metrics
Convert raw W&B run histories into tidy, analysis-ready pandas DataFrames.

## 7. Early convergence
Compare loss and accuracy trajectories across the first five training epochs.

## 8. Learning-rate analysis
Compare learning rates against model convergence speeds and training instability.

## 9. Supervised analysis
Analyze training vs. validation accuracy and loss curves.

## 10. SSL analysis
Analyze self-supervised learning metrics such as contrastive loss and contrastive accuracy.

## 11. Sweep comparisons
Compare different model and hyperparameter configurations within each active sweep.

## 12. Failure detection
Identify anomalous behaviors: NaN values, exploding loss, optimization divergence, or incomplete runs.

## 13. Export
Export compiled DataFrames to CSV and generate plots for the experiment report.